# `expansion_hunter_06_progress` — slide figure

Run **in a VWB JupyterLab app**. Read-only: it describes Batch jobs and lists
GCS objects, and it does not submit or write the ledger.

The figure cell is the screenshot. It shows minicram and genotype as
done / running / failed / not started, the fully genotyped fraction, and
in-flight vCPU against the E2 quota. PNG and PDF land in
`expansion_hunter_rw_scratch/figures/`.

Done means the expected objects exist. Running includes tasks still queued
inside a live job. Not started includes samples that were never submitted and
tasks Batch never started. Failed means the task ran (or finished without
outputs).


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "dispatch.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "dispatch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, gcs, monitor, poll, registry, wave  # noqa: E402
from vwb_batch import plots  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
FIG_DIR = SCRATCH / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", os.environ.get("BATCH_REGION", "us-central1"))
RUN_ID = os.environ.get("EH_RUN_ID", "eh-260929-pilot").strip()
if not RUN_ID:
    raise SystemExit("set RUN_ID / EH_RUN_ID")
OUT_PREFIX = os.environ.get(
    "EH_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
)
E2_CPUS_QUOTA = int(os.environ.get("EH_E2_CPUS_QUOTA", "2400"))

PATHS = registry.run_paths(
    output_bucket=OUTPUT_BUCKET_GS.rstrip("/"),
    pipeline="expansion_hunter",
    run_id=RUN_ID,
)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_ID:", RUN_ID)
print("ledger:", PATHS.prefix)
print("OUT_PREFIX:", OUT_PREFIX)
print("figures:", FIG_DIR)
print("gcloud:", shutil.which("gcloud"))


## Progress figure

Re-run this cell for a fresh screenshot. The following cell lists failed
sample ids and in-flight jobs; leave it out of the slide.


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
project, sa = alloc.require_vwb_env()
if not gcs.exists(PATHS.keep_csv):
    raise SystemExit(f"missing keep.csv: {PATHS.keep_csv}")

local_keep = SCRATCH / f"{RUN_ID}.keep.csv"
gcs.download_file(PATHS.keep_csv, local_keep)
keep_records = cohort.read_csv(local_keep)
sample_ids = [r["sample_id"] for r in keep_records]
events = registry.load_events(PATHS)
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
run_meta = registry.load_run_json(PATHS)
catalog_uri = str(run_meta.get("catalog") or "")
catalog = catalog_uri.rstrip("/").rsplit("/", 1)[-1] or "catalog"
updated_at = registry.utcnow()
print("keep:", len(keep_records), "objects:", len(listing), "catalog:", catalog)

polls: dict = {}
seen: set = set()
for ev in events:
    if ev.get("event") != "job_submitted":
        continue
    job_id = ev.get("job_id")
    if not job_id or job_id in seen:
        continue
    seen.add(job_id)
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("poll failed", job_id, str(exc).splitlines()[0][:160])
        polls[job_id] = {"state": "UNKNOWN", "tasks": []}
        continue
    polls[job_id] = {"state": polled["state"], "tasks": polled["tasks"]}


def objects_done(stage: str) -> set[str]:
    return {
        sid
        for sid in sample_ids
        if poll.objects_ok(eh.expected_objects(sid, stage, OUT_PREFIX), listing)
    }


def categories_for(stage: str, done_ids: set[str]) -> dict[str, str]:
    return wave.classify_stage(
        sample_ids,
        submissions=wave.latest_submissions(events, stage),
        polls=polls,
        done_ids=done_ids,
    )


mini_cat = categories_for("minicram", objects_done("minicram"))
gt_cat = categories_for("genotype", objects_done("genotype"))
counts = {
    "minicram": wave.count_categories(mini_cat),
    "genotype": wave.count_categories(gt_cat),
}
for stage, cats in counts.items():
    print(
        f"{stage}: done {cats['done']}  running {cats['running']}  "
        f"failed {cats['failed']}  not started {cats['not_started']}"
    )

job_rows = monitor.list_summaries(
    project=project,
    region=REGION,
    run_id=RUN_ID,
    pipeline="expansion-hunter",
    enrich_active=True,
)
active = [row for row in job_rows if row.get("state") in monitor.ACTIVE]
paths = plots.plot_cohort_progress(
    FIG_DIR,
    run_id=RUN_ID,
    catalog=catalog,
    counts=counts,
    fully_done=counts["genotype"]["done"],
    keep_n=len(sample_ids),
    inflight_vcpu=monitor.total_inflight_vcpu(job_rows),
    e2_quota=E2_CPUS_QUOTA,
    n_active_jobs=len(active),
    updated_at=updated_at,
)
print("wrote:")
for path in paths:
    print(" ", path)


## Failures and in-flight jobs

Not part of the screenshot. Failed ids are samples whose last task actually
failed or finished without the expected objects.


In [ ]:
def failed_ids(cats: dict[str, str]) -> list[str]:
    return [sid for sid in sample_ids if cats.get(sid) == "failed"]


for stage, cats in (("minicram", mini_cat), ("genotype", gt_cat)):
    failed = failed_ids(cats)
    print(f"{stage} failed: {len(failed)}")
    for sid in failed[:30]:
        print(" ", sid)
    if len(failed) > 30:
        print(f"  ... {len(failed) - 30} more")

print()
print(monitor.format_table(active) if active else "(no jobs in flight)")
